# 03 - Feature Engineering

This notebook turns the cleaned NFL data into features that can be used throughout the rest of the projection model.

The goal is to describe how teams performed, how their rosters were built, and what changed from one season to the next. These features will later be used for player projections, team strength ratings, and game predictions.

In [330]:
from pathlib import Path

import numpy as np
import polars as pl

PROJECT_ROOT = Path.cwd().parent
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

In [331]:
schedule = pl.read_parquet(
    PROCESSED_DIR / "schedule_clean.parquet"
)

print(schedule.shape)
schedule.head()

(2895, 16)


game_id,season,week,gameday,weekday,gametime,away_team,away_score,home_team,home_score,location,result,total,overtime,away_rest,home_rest
str,i32,i32,str,str,str,str,i32,str,i32,str,i32,i32,i32,i32,i32
"""2015_01_PIT_NE""",2015,1,"""2015-09-10""","""Thursday""","""20:30""","""PIT""",21,"""NE""",28,"""Home""",7,49,0,7,7
"""2015_01_BAL_DEN""",2015,1,"""2015-09-13""","""Sunday""","""16:25""","""BAL""",13,"""DEN""",19,"""Home""",6,32,0,7,7
"""2015_01_CAR_JAX""",2015,1,"""2015-09-13""","""Sunday""","""13:00""","""CAR""",20,"""JAX""",9,"""Home""",-11,29,0,7,7
"""2015_01_CIN_OAK""",2015,1,"""2015-09-13""","""Sunday""","""16:25""","""CIN""",33,"""LV""",13,"""Home""",-20,46,0,7,7
"""2015_01_CLE_NYJ""",2015,1,"""2015-09-13""","""Sunday""","""13:00""","""CLE""",10,"""NYJ""",31,"""Home""",21,41,0,7,7


# Team Season Performance

I want to start with one row for each team and season.

The schedule data gives us the basic results for every game, which lets me build features like wins, losses, points scored, points allowed, and point differential. This will become the base team-season table that other features can be added to later.

In [332]:
home_games = schedule.select([
    "game_id",
    "season",
    "week",
    pl.col("home_team").alias("team"),
    pl.col("away_team").alias("opponent"),
    pl.col("home_score").alias("points_for"),
    pl.col("away_score").alias("points_against"),
    pl.lit(1).alias("home_game")
])

away_games = schedule.select([
    "game_id",
    "season",
    "week",
    pl.col("away_team").alias("team"),
    pl.col("home_team").alias("opponent"),
    pl.col("away_score").alias("points_for"),
    pl.col("home_score").alias("points_against"),
    pl.lit(0).alias("home_game")
])

team_games = (
    pl.concat([
        home_games,
        away_games
    ])
    .with_columns(
        (pl.col("points_for") - pl.col("points_against"))
        .alias("point_diff")
    )
    .sort([
        "season",
        "week",
        "team"
    ])
)

team_games.head(10)

game_id,season,week,team,opponent,points_for,points_against,home_game,point_diff
str,i32,i32,str,str,i32,i32,i32,i32
"""2015_01_NO_ARI""",2015,1,"""ARI""","""NO""",31,19,1,12
"""2015_01_PHI_ATL""",2015,1,"""ATL""","""PHI""",26,24,1,2
"""2015_01_BAL_DEN""",2015,1,"""BAL""","""DEN""",13,19,0,-6
"""2015_01_IND_BUF""",2015,1,"""BUF""","""IND""",27,14,1,13
"""2015_01_CAR_JAX""",2015,1,"""CAR""","""JAX""",20,9,0,11
"""2015_01_GB_CHI""",2015,1,"""CHI""","""GB""",23,31,1,-8
"""2015_01_CIN_OAK""",2015,1,"""CIN""","""LV""",33,13,0,20
"""2015_01_CLE_NYJ""",2015,1,"""CLE""","""NYJ""",10,31,0,-21
"""2015_01_NYG_DAL""",2015,1,"""DAL""","""NYG""",27,26,1,1


In [333]:
team_games = team_games.with_columns([
    (pl.col("points_for") > pl.col("points_against"))
    .cast(pl.Int8)
    .alias("win"),

    (pl.col("points_for") < pl.col("points_against"))
    .cast(pl.Int8)
    .alias("loss"),

    (pl.col("points_for") == pl.col("points_against"))
    .cast(pl.Int8)
    .alias("tie")
])

team_games.head(10)

game_id,season,week,team,opponent,points_for,points_against,home_game,point_diff,win,loss,tie
str,i32,i32,str,str,i32,i32,i32,i32,i8,i8,i8
"""2015_01_NO_ARI""",2015,1,"""ARI""","""NO""",31,19,1,12,1,0,0
"""2015_01_PHI_ATL""",2015,1,"""ATL""","""PHI""",26,24,1,2,1,0,0
"""2015_01_BAL_DEN""",2015,1,"""BAL""","""DEN""",13,19,0,-6,0,1,0
"""2015_01_IND_BUF""",2015,1,"""BUF""","""IND""",27,14,1,13,1,0,0
"""2015_01_CAR_JAX""",2015,1,"""CAR""","""JAX""",20,9,0,11,1,0,0
"""2015_01_GB_CHI""",2015,1,"""CHI""","""GB""",23,31,1,-8,0,1,0
"""2015_01_CIN_OAK""",2015,1,"""CIN""","""LV""",33,13,0,20,1,0,0
"""2015_01_CLE_NYJ""",2015,1,"""CLE""","""NYJ""",10,31,0,-21,0,1,0
"""2015_01_NYG_DAL""",2015,1,"""DAL""","""NYG""",27,26,1,1,1,0,0


In [334]:
team_season = (
    team_games
    .group_by([
        "season",
        "team"
    ])
    .agg([
        pl.len().alias("games"),
        pl.col("win").sum().alias("wins"),
        pl.col("loss").sum().alias("losses"),
        pl.col("tie").sum().alias("ties"),
        pl.col("points_for").sum().alias("points_for"),
        pl.col("points_against").sum().alias("points_against"),
        pl.col("point_diff").sum().alias("point_diff")
    ])
    .with_columns([
        (
            (pl.col("wins") + 0.5 * pl.col("ties")) /
            pl.col("games")
        ).alias("win_pct"),

        (
            pl.col("points_for") /
            pl.col("games")
        ).alias("points_for_per_game"),

        (
            pl.col("points_against") /
            pl.col("games")
        ).alias("points_against_per_game"),

        (
            pl.col("point_diff") /
            pl.col("games")
        ).alias("point_diff_per_game")
    ])
    .sort([
        "season",
        "team"
    ])
)

team_season.head(10)

season,team,games,wins,losses,ties,points_for,points_against,point_diff,win_pct,points_for_per_game,points_against_per_game,point_diff_per_game
i32,str,u32,i64,i64,i64,i32,i32,i32,f64,f64,f64,f64
2015,"""ARI""",16,13,3,0,489,313,176,0.8125,30.5625,19.5625,11.0
2015,"""ATL""",16,8,8,0,339,345,-6,0.5,21.1875,21.5625,-0.375
2015,"""BAL""",16,5,11,0,328,401,-73,0.3125,20.5,25.0625,-4.5625
2015,"""BUF""",16,8,8,0,379,359,20,0.5,23.6875,22.4375,1.25
2015,"""CAR""",16,15,1,0,500,308,192,0.9375,31.25,19.25,12.0
2015,"""CHI""",16,6,10,0,335,397,-62,0.375,20.9375,24.8125,-3.875
2015,"""CIN""",16,12,4,0,419,279,140,0.75,26.1875,17.4375,8.75
2015,"""CLE""",16,3,13,0,278,432,-154,0.1875,17.375,27.0,-9.625
2015,"""DAL""",16,4,12,0,275,374,-99,0.25,17.1875,23.375,-6.1875


In [335]:
team_season = team_season.with_columns([
    (
        pl.col("wins") - pl.col("losses")
    ).alias("win_loss_diff"),

    (
        pl.col("point_diff") > 0
    ).cast(pl.Int8).alias("positive_point_diff")
])

## One Score Games

A team's record in close games can make its overall record look better or worse than its underlying performance. I want to track one score games separately so I can later test how much that performance carries over from year to year.

In [336]:
team_games = team_games.with_columns(
    (pl.col("point_diff").abs() <= 8)
    .cast(pl.Int8)
    .alias("one_score_game")
)

one_score_summary = (
    team_games
    .filter(pl.col("one_score_game") == 1)
    .group_by([
        "season",
        "team"
    ])
    .agg([
        pl.len().alias("one_score_games"),
        pl.col("win").sum().alias("one_score_wins"),
        pl.col("loss").sum().alias("one_score_losses"),
        pl.col("tie").sum().alias("one_score_ties")
    ])
    .with_columns(
        (
            (
                pl.col("one_score_wins") +
                0.5 * pl.col("one_score_ties")
            ) /
            pl.col("one_score_games")
        ).alias("one_score_win_pct")
    )
)

In [337]:
team_season = (
    team_season
    .join(
        one_score_summary,
        on=[
            "season",
            "team"
        ],
        how="left"
    )
    .with_columns([
        pl.col("one_score_games").fill_null(0),
        pl.col("one_score_wins").fill_null(0),
        pl.col("one_score_losses").fill_null(0),
        pl.col("one_score_ties").fill_null(0),
        pl.col("one_score_win_pct").fill_null(0)
    ])
)

## Play-by-Play Efficiency

Final scores can hide a lot of what actually happened during a game, so I also want to measure how efficiently each team played on a snap-by-snap basis.

EPA and success rate give us a better idea of whether an offense or defense was consistently creating value rather than just looking at the final score.

In [338]:
pbp = pl.read_parquet(
    PROCESSED_DIR / "play_by_play_clean.parquet"
)

print(pbp.shape)
print(pbp.columns)

(508914, 51)
['game_id', 'play_id', 'season', 'week', 'home_team', 'away_team', 'posteam', 'defteam', 'qtr', 'down', 'ydstogo', 'yardline_100', 'goal_to_go', 'score_differential', 'drive', 'posteam_score', 'posteam_score_post', 'play_type', 'yards_gained', 'epa', 'success', 'wp', 'wpa', 'pass_attempt', 'rush_attempt', 'qb_dropback', 'qb_scramble', 'sack', 'qb_hit', 'complete_pass', 'interception', 'fumble', 'fumble_lost', 'touchdown', 'pass_touchdown', 'rush_touchdown', 'air_yards', 'yards_after_catch', 'cpoe', 'xpass', 'pass_oe', 'first_down', 'third_down_converted', 'third_down_failed', 'fourth_down_converted', 'fourth_down_failed', 'shotgun', 'fixed_drive', 'fixed_drive_result', 'drive_ended_with_score', 'no_huddle']


In [339]:
offensive_plays = (
    pbp
    .filter(
        pl.col("posteam").is_not_null() &
        pl.col("defteam").is_not_null() &
        pl.col("epa").is_not_null() &
        (
            (pl.col("pass_attempt") == 1) |
            (pl.col("rush_attempt") == 1)
        )
    )
)

offensive_plays.shape

(367199, 51)

In [340]:
offensive_plays = (
    pbp
    .filter(
        pl.col("posteam").is_not_null() &
        pl.col("defteam").is_not_null() &
        pl.col("epa").is_not_null() &
        pl.col("play_type").is_in([
            "pass",
            "run"
        ])
    )
)

In [341]:
offensive_efficiency = (
    offensive_plays
    .group_by([
        "season",
        "posteam"
    ])
    .agg([
        pl.len().alias("off_plays"),
        pl.col("epa").mean().alias("off_epa_per_play"),
        pl.col("success").mean().alias("off_success_rate"),

        pl.col("epa")
        .filter(pl.col("play_type") == "pass")
        .mean()
        .alias("off_pass_epa_per_play"),

        pl.col("epa")
        .filter(pl.col("play_type") == "run")
        .mean()
        .alias("off_rush_epa_per_play")
    ])
    .rename({
        "posteam": "team"
    })
)

In [342]:
defensive_efficiency = (
    offensive_plays
    .group_by([
        "season",
        "defteam"
    ])
    .agg([
        pl.len().alias("def_plays"),
        pl.col("epa").mean().alias("def_epa_per_play_allowed"),
        pl.col("success").mean().alias("def_success_rate_allowed"),

        pl.col("epa")
        .filter(pl.col("play_type") == "pass")
        .mean()
        .alias("def_pass_epa_per_play_allowed"),

        pl.col("epa")
        .filter(pl.col("play_type") == "run")
        .mean()
        .alias("def_rush_epa_per_play_allowed")
    ])
    .rename({
        "defteam": "team"
    })
)

In [343]:
team_season = (
    team_season
    .join(
        offensive_efficiency,
        on=[
            "season",
            "team"
        ],
        how="left"
    )
    .join(
        defensive_efficiency,
        on=[
            "season",
            "team"
        ],
        how="left"
    )
)

## Explosive Plays

EPA measures overall efficiency, but it is also useful to separate teams that consistently create big plays.

I define an explosive play as a pass gaining at least 20 yards or a run gaining at least 10 yards.

In [344]:
explosive_plays = (
    offensive_plays
    .with_columns(
        (
            ((pl.col("play_type") == "pass") & (pl.col("yards_gained") >= 20)) |
            ((pl.col("play_type") == "run") & (pl.col("yards_gained") >= 10))
        )
        .cast(pl.Int8)
        .alias("explosive")
    )
)

In [345]:
offensive_explosiveness = (
    explosive_plays
    .group_by([
        "season",
        "posteam"
    ])
    .agg([
        pl.col("explosive").mean().alias("off_explosive_rate"),

        pl.col("explosive")
        .filter(pl.col("play_type") == "pass")
        .mean()
        .alias("off_explosive_pass_rate"),

        pl.col("explosive")
        .filter(pl.col("play_type") == "run")
        .mean()
        .alias("off_explosive_rush_rate")
    ])
    .rename({
        "posteam": "team"
    })
)

In [346]:
defensive_explosiveness = (
    explosive_plays
    .group_by([
        "season",
        "defteam"
    ])
    .agg([
        pl.col("explosive").mean().alias("def_explosive_rate_allowed"),

        pl.col("explosive")
        .filter(pl.col("play_type") == "pass")
        .mean()
        .alias("def_explosive_pass_rate_allowed"),

        pl.col("explosive")
        .filter(pl.col("play_type") == "run")
        .mean()
        .alias("def_explosive_rush_rate_allowed")
    ])
    .rename({
        "defteam": "team"
    })
)

In [347]:
team_season = (
    team_season
    .join(
        offensive_explosiveness,
        on=["season", "team"],
        how="left"
    )
    .join(
        defensive_explosiveness,
        on=["season", "team"],
        how="left"
    )
)

In [348]:
team_season.select([
    "season",
    "team",
    "off_explosive_rate",
    "def_explosive_rate_allowed"
]).filter(
    pl.col("season") == 2025
).sort(
    "off_explosive_rate",
    descending=True
)

season,team,off_explosive_rate,def_explosive_rate_allowed
i32,str,f64,f64
2025,"""BUF""",0.121127,0.108421
2025,"""NE""",0.120944,0.086957
2025,"""BAL""",0.119247,0.103064
2025,"""LA""",0.118371,0.086142
2025,"""CHI""",0.115876,0.117417
…,…,…,…
2025,"""NYJ""",0.078295,0.115859
2025,"""ARI""",0.078212,0.108262
2025,"""CLE""",0.07393,0.089537


## Turnovers and Pressure

Turnovers can swing games quickly, but they are also one of the more volatile parts of team performance. I want to track them separately instead of letting them get mixed into the rest of the efficiency metrics.

Pressure related stats can also help show whether a defense is consistently affecting the quarterback even when it does not always result in a sack.

In [349]:
offensive_turnovers = (
    offensive_plays
    .group_by([
        "season",
        "posteam"
    ])
    .agg([
        pl.col("interception").sum().alias("interceptions_thrown"),
        pl.col("fumble_lost").sum().alias("fumbles_lost"),
        (
            pl.col("interception").sum() +
            pl.col("fumble_lost").sum()
        ).alias("turnovers")
    ])
    .rename({
        "posteam": "team"
    })
)

In [350]:
defensive_turnovers = (
    offensive_plays
    .group_by([
        "season",
        "defteam"
    ])
    .agg([
        pl.col("interception").sum().alias("def_interceptions"),
        pl.col("fumble_lost").sum().alias("def_fumble_recoveries"),
        (
            pl.col("interception").sum() +
            pl.col("fumble_lost").sum()
        ).alias("takeaways")
    ])
    .rename({
        "defteam": "team"
    })
)

In [351]:
defensive_pressure = (
    offensive_plays
    .filter(pl.col("play_type") == "pass")
    .group_by([
        "season",
        "defteam"
    ])
    .agg([
        pl.len().alias("def_pass_plays"),
        pl.col("sack").sum().alias("sacks"),
        pl.col("qb_hit").sum().alias("qb_hits"),
        pl.col("sack").mean().alias("sack_rate"),
        pl.col("qb_hit").mean().alias("qb_hit_rate")
    ])
    .rename({
        "defteam": "team"
    })
)

In [352]:
team_season = (
    team_season
    .join(
        offensive_turnovers,
        on=["season", "team"],
        how="left"
    )
    .join(
        defensive_turnovers,
        on=["season", "team"],
        how="left"
    )
    .join(
        defensive_pressure,
        on=["season", "team"],
        how="left"
    )
    .with_columns(
        (pl.col("takeaways") - pl.col("turnovers"))
        .alias("turnover_margin")
    )
)

## Situational Efficiency

Overall efficiency is important, but certain situations can have a bigger effect on games.

I want to track how teams perform on third down, fourth down, and near the goal line so those situations can be evaluated separately from overall play-by-play efficiency.

In [353]:
offensive_situational = (
    pbp
    .filter(
        pl.col("posteam").is_not_null() &
        pl.col("defteam").is_not_null()
    )
    .group_by([
        "season",
        "posteam"
    ])
    .agg([
        pl.col("third_down_converted").sum().alias("third_down_conversions"),
        (
            pl.col("third_down_converted").sum() +
            pl.col("third_down_failed").sum()
        ).alias("third_down_attempts"),

        pl.col("fourth_down_converted").sum().alias("fourth_down_conversions"),
        (
            pl.col("fourth_down_converted").sum() +
            pl.col("fourth_down_failed").sum()
        ).alias("fourth_down_attempts")
    ])
    .with_columns([
        (
            pl.col("third_down_conversions") /
            pl.col("third_down_attempts")
        ).alias("off_third_down_rate"),

        (
            pl.col("fourth_down_conversions") /
            pl.col("fourth_down_attempts")
        ).alias("off_fourth_down_rate")
    ])
    .rename({
        "posteam": "team"
    })
)

In [354]:
defensive_situational = (
    pbp
    .filter(
        pl.col("posteam").is_not_null() &
        pl.col("defteam").is_not_null()
    )
    .group_by([
        "season",
        "defteam"
    ])
    .agg([
        pl.col("third_down_converted").sum().alias("third_down_conversions_allowed"),
        (
            pl.col("third_down_converted").sum() +
            pl.col("third_down_failed").sum()
        ).alias("third_down_attempts_faced"),

        pl.col("fourth_down_converted").sum().alias("fourth_down_conversions_allowed"),
        (
            pl.col("fourth_down_converted").sum() +
            pl.col("fourth_down_failed").sum()
        ).alias("fourth_down_attempts_faced")
    ])
    .with_columns([
        (
            pl.col("third_down_conversions_allowed") /
            pl.col("third_down_attempts_faced")
        ).alias("def_third_down_rate_allowed"),

        (
            pl.col("fourth_down_conversions_allowed") /
            pl.col("fourth_down_attempts_faced")
        ).alias("def_fourth_down_rate_allowed")
    ])
    .rename({
        "defteam": "team"
    })
)

In [355]:
team_season = (
    team_season
    .join(
        offensive_situational,
        on=["season", "team"],
        how="left"
    )
    .join(
        defensive_situational,
        on=["season", "team"],
        how="left"
    )
)

In [356]:
red_zone_plays = (
    offensive_plays
    .filter(
        pl.col("yardline_100") <= 20
    )
)

In [357]:
offensive_red_zone = (
    red_zone_plays
    .group_by([
        "season",
        "posteam"
    ])
    .agg([
        pl.len().alias("off_red_zone_plays"),
        pl.col("epa").mean().alias("off_red_zone_epa_per_play"),
        pl.col("success").mean().alias("off_red_zone_success_rate")
    ])
    .rename({
        "posteam": "team"
    })
)

defensive_red_zone = (
    red_zone_plays
    .group_by([
        "season",
        "defteam"
    ])
    .agg([
        pl.len().alias("def_red_zone_plays"),
        pl.col("epa").mean().alias("def_red_zone_epa_per_play_allowed"),
        pl.col("success").mean().alias("def_red_zone_success_rate_allowed")
    ])
    .rename({
        "defteam": "team"
    })
)

In [358]:
team_season = (
    team_season
    .join(
        offensive_red_zone,
        on=["season", "team"],
        how="left"
    )
    .join(
        defensive_red_zone,
        on=["season", "team"],
        how="left"
    )
)

In [359]:
team_season.select([
    "season",
    "team",
    "off_red_zone_plays",
    "off_red_zone_epa_per_play",
    "off_red_zone_success_rate",
    "def_red_zone_epa_per_play_allowed"
]).filter(
    pl.col("season") == 2025
).sort(
    "off_red_zone_epa_per_play",
    descending=True
)

season,team,off_red_zone_plays,off_red_zone_epa_per_play,off_red_zone_success_rate,def_red_zone_epa_per_play_allowed
i32,str,u32,f64,f64,f64
2025,"""SF""",201,0.186114,0.462687,-0.075591
2025,"""IND""",194,0.158664,0.453608,-0.096994
2025,"""DEN""",170,0.149269,0.447059,-0.155656
2025,"""DET""",191,0.149179,0.408377,-0.050149
2025,"""BUF""",199,0.103244,0.482412,0.103826
…,…,…,…,…,…
2025,"""NO""",126,-0.143435,0.357143,-0.061601
2025,"""BAL""",177,-0.150634,0.372881,-0.186504
2025,"""NYJ""",129,-0.153498,0.418605,0.243435


# Roster Continuity

Team performance can change quickly when a large part of the roster turns over.

I want to measure how much playing time returns from the previous season, both overall and by position group. This should give the model a better idea of which teams are keeping the same core together and which teams are replacing a large number of contributors.

In [360]:
snap_counts = pl.read_parquet(
    PROCESSED_DIR / "snap_counts_clean.parquet"
)

print(snap_counts.shape)
print(snap_counts.columns)

(264774, 14)
['game_id', 'season', 'week', 'player', 'pfr_player_id', 'position', 'team', 'opponent', 'offense_snaps', 'offense_pct', 'defense_snaps', 'defense_pct', 'st_snaps', 'st_pct']


In [361]:
player_snaps = (
    snap_counts
    .group_by([
        "season",
        "team",
        "pfr_player_id",
        "player",
        "position"
    ])
    .agg([
        pl.col("offense_snaps").sum().alias("offense_snaps"),
        pl.col("defense_snaps").sum().alias("defense_snaps")
    ])
    .with_columns(
        (
            pl.col("offense_snaps") +
            pl.col("defense_snaps")
        ).alias("total_snaps")
    )
)

In [362]:
previous_snaps = (
    player_snaps
    .select([
        pl.col("season") + 1,
        "team",
        "pfr_player_id",
        pl.col("offense_snaps").alias("previous_offense_snaps"),
        pl.col("defense_snaps").alias("previous_defense_snaps"),
        pl.col("total_snaps").alias("previous_total_snaps")
    ])
)

In [363]:
returning_players = (
    player_snaps
    .join(
        previous_snaps,
        on=[
            "season",
            "team",
            "pfr_player_id"
        ],
        how="left"
    )
    .with_columns([
        pl.col("previous_offense_snaps").fill_null(0),
        pl.col("previous_defense_snaps").fill_null(0),
        pl.col("previous_total_snaps").fill_null(0)
    ])
)

In [364]:
roster_continuity = (
    returning_players
    .filter(pl.col("season") >= 2016)
    .group_by([
        "season",
        "team"
    ])
    .agg([
        pl.col("previous_offense_snaps").sum().alias("returning_offense_snaps"),
        pl.col("previous_defense_snaps").sum().alias("returning_defense_snaps"),
        pl.col("previous_total_snaps").sum().alias("returning_total_snaps")
    ])
)

In [365]:
previous_team_snaps = (
    player_snaps
    .group_by([
        "season",
        "team"
    ])
    .agg([
        pl.col("offense_snaps").sum().alias("team_offense_snaps"),
        pl.col("defense_snaps").sum().alias("team_defense_snaps"),
        pl.col("total_snaps").sum().alias("team_total_snaps")
    ])
    .with_columns(
        pl.col("season") + 1
    )
)

In [366]:
roster_continuity = (
    roster_continuity
    .join(
        previous_team_snaps,
        on=[
            "season",
            "team"
        ],
        how="left"
    )
    .with_columns([
        (
            pl.col("returning_offense_snaps") /
            pl.col("team_offense_snaps")
        ).alias("offense_continuity"),

        (
            pl.col("returning_defense_snaps") /
            pl.col("team_defense_snaps")
        ).alias("defense_continuity"),

        (
            pl.col("returning_total_snaps") /
            pl.col("team_total_snaps")
        ).alias("overall_continuity")
    ])
)

In [367]:
team_season = (
    team_season
    .join(
        roster_continuity.select([
            "season",
            "team",
            "offense_continuity",
            "defense_continuity",
            "overall_continuity"
        ]),
        on=[
            "season",
            "team"
        ],
        how="left"
    )
)

### Position Group Continuity

Overall continuity does not capture where roster turnover happened. Returning players at quarterback or along the offensive line can mean something different than returning the same amount of playing time at other positions.

I separate a few major position groups so the model can account for where teams experienced the most change.

In [368]:
player_snaps = (
    player_snaps
    .with_columns(
        pl.when(pl.col("position") == "QB")
        .then(pl.lit("QB"))

        .when(pl.col("position").is_in([
            "C", "C/G", "G", "G/C", "G/OT", "G/T",
            "OG", "OL", "OT", "T", "T/G"
        ]))
        .then(pl.lit("OL"))

        .when(pl.col("position").is_in([
            "RB", "RB/F", "RB/W", "HB",
            "FB", "FB/D", "FB/R", "FB/T",
            "WR", "WR/R", "TE", "TE/D"
        ]))
        .then(pl.lit("Skill"))

        .when(pl.col("position").is_in([
            "DE", "DE/D", "DE/L", "DL", "DT", "DT/D",
            "NT", "LB", "LB/F", "ILB", "MLB", "OLB"
        ]))
        .then(pl.lit("Front Seven"))

        .when(pl.col("position").is_in([
            "CB", "CB/R", "DB", "DB/L",
            "FS", "S", "SS"
        ]))
        .then(pl.lit("Secondary"))

        .otherwise(None)
        .alias("position_group")
    )
)

In [369]:
position_group_snaps = (
    player_snaps
    .filter(
        pl.col("position_group").is_not_null()
    )
    .group_by([
        "season",
        "team",
        "pfr_player_id",
        "position_group"
    ])
    .agg(
        pl.col("total_snaps")
        .sum()
        .alias("total_snaps")
    )
)

In [370]:
previous_position_snaps = (
    position_group_snaps
    .select([
        pl.col("season") + 1,
        "team",
        "pfr_player_id",
        "position_group",
        pl.col("total_snaps").alias("previous_position_snaps")
    ])
)

In [371]:
returning_position_snaps = (
    position_group_snaps
    .join(
        previous_position_snaps,
        on=[
            "season",
            "team",
            "pfr_player_id",
            "position_group"
        ],
        how="left"
    )
    .with_columns(
        pl.col("previous_position_snaps").fill_null(0)
    )
)

In [372]:
previous_position_totals = (
    position_group_snaps
    .group_by([
        "season",
        "team",
        "position_group"
    ])
    .agg(
        pl.col("total_snaps")
        .sum()
        .alias("position_snaps")
    )
    .with_columns(
        pl.col("season") + 1
    )
)

In [373]:
position_continuity = (
    returning_position_snaps
    .filter(pl.col("season") >= 2016)
    .group_by([
        "season",
        "team",
        "position_group"
    ])
    .agg(
        pl.col("previous_position_snaps")
        .sum()
        .alias("returning_position_snaps")
    )
    .join(
        previous_position_totals,
        on=[
            "season",
            "team",
            "position_group"
        ],
        how="left"
    )
    .with_columns(
        (
            pl.col("returning_position_snaps") /
            pl.col("position_snaps")
        ).alias("continuity")
    )
)

In [374]:
position_continuity_wide = (
    position_continuity
    .select([
        "season",
        "team",
        "position_group",
        "continuity"
    ])
    .pivot(
        values="continuity",
        index=["season", "team"],
        on="position_group"
    )
    .rename({
        "QB": "qb_continuity",
        "OL": "ol_continuity",
        "Skill": "skill_continuity",
        "Front Seven": "front_seven_continuity",
        "Secondary": "secondary_continuity"
    })
)

In [375]:
team_season = (
    team_season
    .join(
        position_continuity_wide,
        on=["season", "team"],
        how="left"
    )
)

In [376]:
team_season.select([
    "season",
    "team",
    "qb_continuity",
    "ol_continuity",
    "skill_continuity",
    "front_seven_continuity",
    "secondary_continuity"
]).filter(
    pl.col("season") == 2025
).head(10)

season,team,qb_continuity,ol_continuity,skill_continuity,front_seven_continuity,secondary_continuity
i32,str,f64,f64,f64,f64,f64
2025,"""ARI""",0.971715,0.895176,0.971722,0.512751,0.706805
2025,"""ATL""",1.0,0.696641,0.976549,0.492808,0.75947
2025,"""BAL""",0.967568,0.813037,0.913524,0.860389,0.592425
2025,"""BUF""",0.981651,0.987496,0.735756,0.732124,0.79177
2025,"""CAR""",1.0,0.999809,0.678102,0.438638,0.566495
2025,"""CHI""",1.0,0.324149,0.726245,0.678518,0.955373
2025,"""CIN""",1.0,0.57914,0.831169,0.57506,0.736045
2025,"""CLE""",0.0,0.787584,0.512434,0.551913,0.62069
2025,"""DAL""",0.427839,0.817096,0.753717,0.442504,0.709029


# Player Production

Roster continuity tells me who came back, but it does not tell me how productive those players were.

I want to summarize player production by team and season so I can later measure how much offensive value, usage, and experience each roster is carrying forward.

In [377]:
player_stats = pl.read_parquet(
    PROCESSED_DIR / "player_stats_clean.parquet"
)

print(player_stats.shape)
print(player_stats.columns)

(21366, 148)
['player_id', 'player_name', 'player_display_name', 'position', 'position_group', 'headshot_url', 'season', 'season_type', 'recent_team', 'games', 'completions', 'attempts', 'passing_yards', 'passing_tds', 'passing_interceptions', 'sacks_suffered', 'sack_yards_lost', 'sack_fumbles', 'sack_fumbles_lost', 'passing_air_yards', 'passing_yards_after_catch', 'passing_first_downs', 'passing_epa', 'passing_cpoe', 'passing_2pt_conversions', 'pacr', 'passing_10', 'passing_16', 'passing_20', 'passing_40', 'carries', 'rushing_yards', 'rushing_tds', 'rushing_fumbles', 'rushing_fumbles_lost', 'rushing_first_downs', 'rushing_epa', 'rushing_2pt_conversions', 'rushing_10', 'rushing_12', 'rushing_20', 'rushing_40', 'receptions', 'targets', 'receiving_yards', 'receiving_tds', 'receiving_fumbles', 'receiving_fumbles_lost', 'receiving_air_yards', 'receiving_yards_after_catch', 'receiving_first_downs', 'receiving_epa', 'receiving_2pt_conversions', 'receiving_10', 'receiving_16', 'receiving_20',

In [378]:
player_production = (
    player_stats
    .select([
        "player_id",
        "player_name",
        "player_display_name",
        "position",
        "position_group",
        "season",
        pl.col("recent_team").alias("team"),
        "games",

        "attempts",
        "completions",
        "passing_yards",
        "passing_tds",
        "passing_interceptions",
        "sacks_suffered",
        "passing_epa",
        "passing_cpoe",

        "carries",
        "rushing_yards",
        "rushing_tds",
        "rushing_fumbles_lost",
        "rushing_epa",

        "receptions",
        "targets",
        "receiving_yards",
        "receiving_tds",
        "receiving_fumbles_lost",
        "receiving_epa",
        "target_share",
        "air_yards_share",

        "def_tackles_solo",
        "def_tackle_assists",
        "def_tackles_for_loss",
        "def_fumbles_forced",
        "def_sacks",
        "def_qb_hits",
        "def_interceptions",
        "def_pass_defended"
    ])
    .sort([
        "season",
        "team",
        "player_id"
    ])
)

In [379]:
player_production = (
    player_production
    .with_columns([
        (pl.col("passing_yards") / pl.col("games"))
        .alias("passing_yards_per_game"),

        (pl.col("rushing_yards") / pl.col("games"))
        .alias("rushing_yards_per_game"),

        (pl.col("receiving_yards") / pl.col("games"))
        .alias("receiving_yards_per_game"),

        (pl.col("targets") / pl.col("games"))
        .alias("targets_per_game"),

        pl.when(pl.col("attempts") > 0)
        .then(pl.col("completions") / pl.col("attempts"))
        .otherwise(None)
        .alias("completion_rate"),

        pl.when(pl.col("attempts") > 0)
        .then(pl.col("passing_interceptions") / pl.col("attempts"))
        .otherwise(None)
        .alias("interception_rate"),

        pl.when(pl.col("carries") > 0)
        .then(pl.col("rushing_yards") / pl.col("carries"))
        .otherwise(None)
        .alias("yards_per_carry"),

        pl.when(pl.col("targets") > 0)
        .then(pl.col("receptions") / pl.col("targets"))
        .otherwise(None)
        .alias("catch_rate"),

        pl.when(pl.col("targets") > 0)
        .then(pl.col("receiving_yards") / pl.col("targets"))
        .otherwise(None)
        .alias("yards_per_target")
    ])
)

In [380]:
print("Rows:", player_production.height)

print(
    "Duplicate player-seasons:",
    player_production
    .group_by([
        "season",
        "player_id"
    ])
    .len()
    .filter(pl.col("len") > 1)
    .height
)

print(
    "Infinite rate values:",
    player_production
    .select([
        pl.col("completion_rate").is_infinite().sum(),
        pl.col("interception_rate").is_infinite().sum(),
        pl.col("yards_per_carry").is_infinite().sum(),
        pl.col("catch_rate").is_infinite().sum(),
        pl.col("yards_per_target").is_infinite().sum()
    ])
)

Rows: 21366
Duplicate player-seasons: 0
Infinite rate values: shape: (1, 5)
┌─────────────────┬───────────────────┬─────────────────┬────────────┬──────────────────┐
│ completion_rate ┆ interception_rate ┆ yards_per_carry ┆ catch_rate ┆ yards_per_target │
│ ---             ┆ ---               ┆ ---             ┆ ---        ┆ ---              │
│ u32             ┆ u32               ┆ u32             ┆ u32        ┆ u32              │
╞═════════════════╪═══════════════════╪═════════════════╪════════════╪══════════════════╡
│ 0               ┆ 0                 ┆ 0               ┆ 0          ┆ 0                │
└─────────────────┴───────────────────┴─────────────────┴────────────┴──────────────────┘


### Returning Production

Snap continuity measures how much playing time a team brings back, but it does not account for how productive those players were.

I also want to track how much of a team's previous season production returns the following year. This gives another way to measure how much proven production a roster keeps or loses.

In [381]:
previous_offensive_production = (
    player_production
    .select([
        pl.col("season") + 1,
        "team",
        "player_id",

        pl.col("passing_yards")
        .clip(lower_bound=0)
        .alias("previous_passing_yards"),

        pl.col("rushing_yards")
        .clip(lower_bound=0)
        .alias("previous_rushing_yards"),

        pl.col("receiving_yards")
        .clip(lower_bound=0)
        .alias("previous_receiving_yards")
    ])
)

In [382]:
returning_offensive_production = (
    player_production
    .select([
        "season",
        "team",
        "player_id"
    ])
    .join(
        previous_offensive_production,
        on=[
            "season",
            "team",
            "player_id"
        ],
        how="left"
    )
    .with_columns([
        pl.col("previous_passing_yards").fill_null(0),
        pl.col("previous_rushing_yards").fill_null(0),
        pl.col("previous_receiving_yards").fill_null(0)
    ])
)

In [383]:
previous_team_production = (
    player_production
    .group_by([
        "season",
        "team"
    ])
    .agg([
        pl.col("passing_yards")
        .clip(lower_bound=0)
        .sum()
        .alias("team_passing_yards"),

        pl.col("rushing_yards")
        .clip(lower_bound=0)
        .sum()
        .alias("team_rushing_yards"),

        pl.col("receiving_yards")
        .clip(lower_bound=0)
        .sum()
        .alias("team_receiving_yards")
    ])
    .with_columns(
        pl.col("season") + 1
    )
)

In [384]:
returning_team_production = (
    returning_offensive_production
    .filter(pl.col("season") >= 2016)
    .group_by([
        "season",
        "team"
    ])
    .agg([
        pl.col("previous_passing_yards").sum().alias("returning_passing_yards"),
        pl.col("previous_rushing_yards").sum().alias("returning_rushing_yards"),
        pl.col("previous_receiving_yards").sum().alias("returning_receiving_yards")
    ])
    .join(
        previous_team_production,
        on=["season", "team"],
        how="left"
    )
    .with_columns([
        (
            pl.col("returning_passing_yards") /
            pl.col("team_passing_yards")
        ).alias("passing_yards_continuity"),

        (
            pl.col("returning_rushing_yards") /
            pl.col("team_rushing_yards")
        ).alias("rushing_yards_continuity"),

        (
            pl.col("returning_receiving_yards") /
            pl.col("team_receiving_yards")
        ).alias("receiving_yards_continuity")
    ])
)

In [385]:
print(
    returning_team_production.select([
        pl.col("passing_yards_continuity").min().alias("passing_min"),
        pl.col("passing_yards_continuity").max().alias("passing_max"),

        pl.col("rushing_yards_continuity").min().alias("rushing_min"),
        pl.col("rushing_yards_continuity").max().alias("rushing_max"),

        pl.col("receiving_yards_continuity").min().alias("receiving_min"),
        pl.col("receiving_yards_continuity").max().alias("receiving_max")
    ])
)

shape: (1, 6)
┌─────────────┬─────────────┬─────────────┬─────────────┬───────────────┬───────────────┐
│ passing_min ┆ passing_max ┆ rushing_min ┆ rushing_max ┆ receiving_min ┆ receiving_max │
│ ---         ┆ ---         ┆ ---         ┆ ---         ┆ ---           ┆ ---           │
│ f64         ┆ f64         ┆ f64         ┆ f64         ┆ f64           ┆ f64           │
╞═════════════╪═════════════╪═════════════╪═════════════╪═══════════════╪═══════════════╡
│ 0.0         ┆ 1.0         ┆ 0.065678    ┆ 1.0         ┆ 0.161219      ┆ 0.99715       │
└─────────────┴─────────────┴─────────────┴─────────────┴───────────────┴───────────────┘


In [386]:
team_season = (
    team_season
    .join(
        returning_team_production.select([
            "season",
            "team",
            "passing_yards_continuity",
            "rushing_yards_continuity",
            "receiving_yards_continuity"
        ]),
        on=["season", "team"],
        how="left"
    )
)

# Injuries and Availability

Roster strength is not just about who is on the team. It also matters how often important players were actually available.

I want to summarize injury and availability information at the team-season level so later models can distinguish between poor performance caused by a weak roster and poor performance caused partly by missed time.

In [387]:
injuries = pl.read_parquet(
    PROCESSED_DIR / "injuries_clean.parquet"
)

print(injuries.shape)
print(injuries.columns)

(58447, 17)
['season', 'game_type', 'team', 'week', 'gsis_id', 'position', 'full_name', 'first_name', 'last_name', 'report_primary_injury', 'report_secondary_injury', 'report_status', 'practice_primary_injury', 'practice_secondary_injury', 'practice_status', 'date_modified', 'season_type']


In [388]:
injury_reports = (
    injuries
    .filter(
        pl.col("report_status").is_not_null()
    )
    .with_columns([
        (pl.col("report_status") == "Out")
        .cast(pl.Int8)
        .alias("out"),

        (pl.col("report_status") == "Doubtful")
        .cast(pl.Int8)
        .alias("doubtful"),

        (pl.col("report_status") == "Questionable")
        .cast(pl.Int8)
        .alias("questionable")
    ])
)

In [389]:
team_injuries = (
    injury_reports
    .group_by([
        "season",
        "team"
    ])
    .agg([
        pl.len().alias("injury_report_entries"),
        pl.col("gsis_id").n_unique().alias("players_on_injury_report"),
        pl.col("out").sum().alias("out_designations"),
        pl.col("doubtful").sum().alias("doubtful_designations"),
        pl.col("questionable").sum().alias("questionable_designations")
    ])
)

In [390]:
print(
    injury_reports["report_status"]
    .value_counts()
    .sort("count", descending=True)
)

print(
    team_injuries.filter(
        pl.col("season") == 2025
    ).sort(
        "out_designations",
        descending=True
    ).head(10)
)

shape: (5, 2)
┌───────────────┬───────┐
│ report_status ┆ count │
│ ---           ┆ ---   │
│ str           ┆ u32   │
╞═══════════════╪═══════╡
│ Questionable  ┆ 15459 │
│ Out           ┆ 11055 │
│ Probable      ┆ 2553  │
│ Doubtful      ┆ 1856  │
│ Note          ┆ 6     │
└───────────────┴───────┘
shape: (10, 7)
┌────────┬──────┬────────────────┬────────────────┬────────────────┬───────────────┬───────────────┐
│ season ┆ team ┆ injury_report_ ┆ players_on_inj ┆ out_designatio ┆ doubtful_desi ┆ questionable_ │
│ ---    ┆ ---  ┆ entries        ┆ ury_report     ┆ ns             ┆ gnations      ┆ designations  │
│ f64    ┆ str  ┆ ---            ┆ ---            ┆ ---            ┆ ---           ┆ ---           │
│        ┆      ┆ u32            ┆ u32            ┆ i64            ┆ i64           ┆ i64           │
╞════════╪══════╪════════════════╪════════════════╪════════════════╪═══════════════╪═══════════════╡
│ 2025.0 ┆ DET  ┆ 149            ┆ 43             ┆ 70             ┆ 2         

In [391]:
injury_reports = (
    injuries
    .filter(
        pl.col("report_status").is_not_null()
    )
    .with_columns([
        pl.col("season")
        .cast(pl.Int32),

        (pl.col("report_status") == "Out")
        .cast(pl.Int8)
        .alias("out"),

        (pl.col("report_status") == "Doubtful")
        .cast(pl.Int8)
        .alias("doubtful"),

        (pl.col("report_status") == "Questionable")
        .cast(pl.Int8)
        .alias("questionable")
    ])
)

In [392]:
team_injuries = (
    injury_reports
    .group_by([
        "season",
        "team"
    ])
    .agg([
        pl.len().alias("injury_report_entries"),
        pl.col("gsis_id").n_unique().alias("players_on_injury_report"),
        pl.col("out").sum().alias("out_designations"),
        pl.col("doubtful").sum().alias("doubtful_designations"),
        pl.col("questionable").sum().alias("questionable_designations")
    ])
)

In [393]:
team_injuries = (
    team_injuries
    .with_columns([
        (
            pl.col("out_designations") /
            pl.col("injury_report_entries")
        ).alias("out_rate"),

        (
            pl.col("doubtful_designations") /
            pl.col("injury_report_entries")
        ).alias("doubtful_rate"),

        (
            pl.col("questionable_designations") /
            pl.col("injury_report_entries")
        ).alias("questionable_rate")
    ])
)

In [394]:
team_season = (
    team_season
    .join(
        team_injuries,
        on=["season", "team"],
        how="left"
    )
)

In [395]:
print("Team-season rows:", team_season.height)

print(
    team_season
    .filter(pl.col("season") == 2025)
    .select([
        "team",
        "injury_report_entries",
        "players_on_injury_report",
        "out_designations",
        "out_rate"
    ])
    .sort(
        "out_designations",
        descending=True
    )
)

print(
    "2025 teams missing injury data:",
    team_season
    .filter(
        (pl.col("season") == 2025) &
        pl.col("injury_report_entries").is_null()
    )
    .height
)

Team-season rows: 352
shape: (32, 5)
┌──────┬───────────────────────┬──────────────────────────┬──────────────────┬──────────┐
│ team ┆ injury_report_entries ┆ players_on_injury_report ┆ out_designations ┆ out_rate │
│ ---  ┆ ---                   ┆ ---                      ┆ ---              ┆ ---      │
│ str  ┆ u32                   ┆ u32                      ┆ i64              ┆ f64      │
╞══════╪═══════════════════════╪══════════════════════════╪══════════════════╪══════════╡
│ DET  ┆ 149                   ┆ 43                       ┆ 70               ┆ 0.469799 │
│ ATL  ┆ 112                   ┆ 43                       ┆ 63               ┆ 0.5625   │
│ ARI  ┆ 139                   ┆ 41                       ┆ 61               ┆ 0.438849 │
│ NYG  ┆ 112                   ┆ 39                       ┆ 58               ┆ 0.517857 │
│ TB   ┆ 87                    ┆ 27                       ┆ 54               ┆ 0.62069  │
│ …    ┆ …                     ┆ …                        ┆ …  

### Injury Impact

Raw injury counts do not tell the full story because missing time from a starter matters more than missing time from a reserve player.

I use player snap totals to give more weight to injury designations involving players who had a larger role on the team.

In [396]:
injury_player_snaps = (
    player_snaps
    .select([
        "season",
        "team",
        pl.col("pfr_player_id").alias("player_id"),
        "total_snaps"
    ])
)

In [397]:
rosters = pl.read_parquet(
    PROCESSED_DIR / "rosters_clean.parquet"
)

print(rosters.shape)
print(rosters.columns)

(33195, 16)
['season', 'team', 'gsis_id', 'full_name', 'football_name', 'position', 'depth_chart_position', 'status', 'years_exp', 'birth_date', 'height', 'weight', 'college', 'espn_id', 'pfr_id', 'sportradar_id']


In [398]:
player_id_bridge = (
    rosters
    .filter(
        pl.col("gsis_id").is_not_null() &
        pl.col("pfr_id").is_not_null()
    )
    .select([
        "season",
        "team",
        "gsis_id",
        "pfr_id"
    ])
    .unique()
)

In [399]:
bridge_duplicates = (
    player_id_bridge
    .group_by([
        "season",
        "team",
        "gsis_id"
    ])
    .agg(
        pl.col("pfr_id").n_unique().alias("pfr_ids")
    )
    .filter(
        pl.col("pfr_ids") > 1
    )
)

print("Conflicting ID mappings:", bridge_duplicates.height)

Conflicting ID mappings: 0


In [400]:
team_injury_weekly = (
    injury_reports
    .group_by([
        "season",
        "team",
        "week"
    ])
    .agg([
        pl.col("out").sum().alias("weekly_out"),
        pl.col("doubtful").sum().alias("weekly_doubtful"),
        pl.col("questionable").sum().alias("weekly_questionable")
    ])
    .group_by([
        "season",
        "team"
    ])
    .agg([
        pl.col("weekly_out").mean().alias("avg_out_per_week"),
        pl.col("weekly_doubtful").mean().alias("avg_doubtful_per_week"),
        pl.col("weekly_questionable").mean().alias("avg_questionable_per_week")
    ])
)

In [401]:
team_season = (
    team_season
    .join(
        team_injury_weekly,
        on=["season", "team"],
        how="left"
    )
)

In [402]:
print("Shape:", team_season.shape)

print(
    team_season
    .null_count()
    .transpose(
        include_header=True,
        header_name="feature",
        column_names=["nulls"]
    )
    .filter(pl.col("nulls") > 0)
    .sort("nulls", descending=True)
)

Shape: (352, 88)
shape: (11, 2)
┌────────────────────────────┬───────┐
│ feature                    ┆ nulls │
│ ---                        ┆ ---   │
│ str                        ┆ u32   │
╞════════════════════════════╪═══════╡
│ offense_continuity         ┆ 32    │
│ defense_continuity         ┆ 32    │
│ overall_continuity         ┆ 32    │
│ secondary_continuity       ┆ 32    │
│ front_seven_continuity     ┆ 32    │
│ …                          ┆ …     │
│ ol_continuity              ┆ 32    │
│ skill_continuity           ┆ 32    │
│ passing_yards_continuity   ┆ 32    │
│ rushing_yards_continuity   ┆ 32    │
│ receiving_yards_continuity ┆ 32    │
└────────────────────────────┴───────┘


In [403]:
null_check = (
    team_season
    .group_by("season")
    .agg([
        pl.col("overall_continuity").null_count()
        .alias("continuity_nulls"),

        pl.col("passing_yards_continuity").null_count()
        .alias("production_nulls"),

        pl.col("injury_report_entries").null_count()
        .alias("injury_nulls")
    ])
    .sort("season")
)

null_check

season,continuity_nulls,production_nulls,injury_nulls
i32,u32,u32,u32
2015,32,32,0
2016,0,0,0
2017,0,0,0
2018,0,0,0
2019,0,0,0
…,…,…,…
2021,0,0,0
2022,0,0,0
2023,0,0,0


In [404]:
TEAM_FEATURES_PATH = PROCESSED_DIR / "team_season_features.parquet"

team_season.write_parquet(TEAM_FEATURES_PATH)

print(f"Saved {team_season.height:,} team-season rows")
print(f"Features: {team_season.width}")
print(f"Path: {TEAM_FEATURES_PATH}")

Saved 352 team-season rows
Features: 88
Path: c:\Users\efriedman\Desktop\NFL-Season-Projections\data\processed\team_season_features.parquet


## Recent Team Performance

A single season can be noisy, so I also want to capture how a team's performance has changed over time.

These features compare current team performance with previous seasons and create recent-performance measures that can later help the model distinguish sustained team strength from one year spikes or declines.

In [405]:
team_season = (
    team_season
    .sort([
        "team",
        "season"
    ])
    .with_columns([
        pl.col("off_epa_per_play")
        .shift(1)
        .over("team")
        .alias("prev_off_epa_per_play"),

        pl.col("def_epa_per_play_allowed")
        .shift(1)
        .over("team")
        .alias("prev_def_epa_per_play_allowed"),

        pl.col("wins")
        .shift(1)
        .over("team")
        .alias("prev_wins")
    ])
)

In [406]:
team_season = (
    team_season
    .with_columns([
        (
            pl.col("off_epa_per_play") -
            pl.col("prev_off_epa_per_play")
        ).alias("off_epa_change"),

        (
            pl.col("def_epa_per_play_allowed") -
            pl.col("prev_def_epa_per_play_allowed")
        ).alias("def_epa_allowed_change"),

        (
            pl.col("wins") -
            pl.col("prev_wins")
        ).alias("wins_change")
    ])
)

In [407]:
print(
    team_season
    .filter(
        pl.col("team") == "BUF"
    )
    .select([
        "season",
        "wins",
        "prev_wins",
        "wins_change",
        "off_epa_per_play",
        "prev_off_epa_per_play",
        "off_epa_change",
        "def_epa_per_play_allowed",
        "prev_def_epa_per_play_allowed",
        "def_epa_allowed_change"
    ])
    .sort("season")
)

shape: (11, 10)
┌────────┬──────┬───────────┬─────────────┬───┬─────────────┬────────────┬────────────┬────────────┐
│ season ┆ wins ┆ prev_wins ┆ wins_change ┆ … ┆ off_epa_cha ┆ def_epa_pe ┆ prev_def_e ┆ def_epa_al │
│ ---    ┆ ---  ┆ ---       ┆ ---         ┆   ┆ nge         ┆ r_play_all ┆ pa_per_pla ┆ lowed_chan │
│ i32    ┆ i64  ┆ i64       ┆ i64         ┆   ┆ ---         ┆ owed       ┆ y_allowed  ┆ ge         │
│        ┆      ┆           ┆             ┆   ┆ f64         ┆ ---        ┆ ---        ┆ ---        │
│        ┆      ┆           ┆             ┆   ┆             ┆ f64        ┆ f64        ┆ f64        │
╞════════╪══════╪═══════════╪═════════════╪═══╪═════════════╪════════════╪════════════╪════════════╡
│ 2015   ┆ 8    ┆ null      ┆ null        ┆ … ┆ null        ┆ 0.030996   ┆ null       ┆ null       │
│ 2016   ┆ 7    ┆ 8         ┆ -1          ┆ … ┆ 0.021067    ┆ 0.018128   ┆ 0.030996   ┆ -0.012868  │
│ 2017   ┆ 9    ┆ 7         ┆ 2           ┆ … ┆ -0.135955   ┆ 0.01525    ┆ 

In [408]:
team_season = (
    team_season
    .sort([
        "team",
        "season"
    ])
    .with_columns([
        pl.col("off_epa_per_play")
        .rolling_mean(window_size=2, min_samples=2)
        .over("team")
        .alias("off_epa_2yr_avg"),

        pl.col("def_epa_per_play_allowed")
        .rolling_mean(window_size=2, min_samples=2)
        .over("team")
        .alias("def_epa_allowed_2yr_avg"),

        pl.col("off_epa_per_play")
        .rolling_mean(window_size=3, min_samples=3)
        .over("team")
        .alias("off_epa_3yr_avg"),

        pl.col("def_epa_per_play_allowed")
        .rolling_mean(window_size=3, min_samples=3)
        .over("team")
        .alias("def_epa_allowed_3yr_avg")
    ])
)

In [409]:
print(
    team_season
    .filter(
        (pl.col("team") == "BUF") &
        (pl.col("season") >= 2023)
    )
    .select([
        "season",
        "off_epa_per_play",
        "off_epa_2yr_avg",
        "off_epa_3yr_avg",
        "def_epa_per_play_allowed",
        "def_epa_allowed_2yr_avg",
        "def_epa_allowed_3yr_avg"
    ])
    .sort("season")
)

shape: (3, 7)
┌────────┬──────────────┬──────────────┬──────────────┬──────────────┬──────────────┬──────────────┐
│ season ┆ off_epa_per_ ┆ off_epa_2yr_ ┆ off_epa_3yr_ ┆ def_epa_per_ ┆ def_epa_allo ┆ def_epa_allo │
│ ---    ┆ play         ┆ avg          ┆ avg          ┆ play_allowed ┆ wed_2yr_avg  ┆ wed_3yr_avg  │
│ i32    ┆ ---          ┆ ---          ┆ ---          ┆ ---          ┆ ---          ┆ ---          │
│        ┆ f64          ┆ f64          ┆ f64          ┆ f64          ┆ f64          ┆ f64          │
╞════════╪══════════════╪══════════════╪══════════════╪══════════════╪══════════════╪══════════════╡
│ 2023   ┆ 0.1083       ┆ 0.112058     ┆ 0.108354     ┆ -0.073917    ┆ -0.063118    ┆ -0.089049    │
│ 2024   ┆ 0.188474     ┆ 0.148387     ┆ 0.13753      ┆ -0.008508    ┆ -0.041212    ┆ -0.044915    │
│ 2025   ┆ 0.135541     ┆ 0.162008     ┆ 0.144105     ┆ -0.016306    ┆ -0.012407    ┆ -0.03291     │
└────────┴──────────────┴──────────────┴──────────────┴──────────────┴───────

## Opponent Strength

Raw team performance does not happen against the same level of competition.

To add context, I measure the average strength of each team's opponents using opponent offensive and defensive efficiency. This gives the model information about the quality of competition behind a team's season level results.

In [410]:
for name in [
    "schedule",
    "games",
    "team_games",
    "schedule_games"
]:
    if name in globals():
        df = globals()[name]

        print(f"\n{name}:")
        print(df.shape)
        print(df.columns)


schedule:
(2895, 16)
['game_id', 'season', 'week', 'gameday', 'weekday', 'gametime', 'away_team', 'away_score', 'home_team', 'home_score', 'location', 'result', 'total', 'overtime', 'away_rest', 'home_rest']

team_games:
(5790, 13)
['game_id', 'season', 'week', 'team', 'opponent', 'points_for', 'points_against', 'home_game', 'point_diff', 'win', 'loss', 'tie', 'one_score_game']


In [411]:
opponent_ratings = (
    team_season
    .select([
        "season",
        pl.col("team").alias("opponent"),
        pl.col("off_epa_per_play").alias("opponent_off_epa"),
        pl.col("def_epa_per_play_allowed").alias("opponent_def_epa_allowed")
    ])
)

In [412]:
team_games_with_opponent_strength = (
    team_games
    .join(
        opponent_ratings,
        on=[
            "season",
            "opponent"
        ],
        how="left"
    )
)

In [413]:
schedule_strength = (
    team_games_with_opponent_strength
    .group_by([
        "season",
        "team"
    ])
    .agg([
        pl.col("opponent_off_epa")
        .mean()
        .alias("avg_opponent_off_epa"),

        pl.col("opponent_def_epa_allowed")
        .mean()
        .alias("avg_opponent_def_epa_allowed")
    ])
)

In [414]:
team_season = (
    team_season
    .join(
        schedule_strength,
        on=[
            "season",
            "team"
        ],
        how="left"
    )
)

### Schedule-Adjusted Efficiency

Opponent strength gives additional context to raw EPA.

I create simple adjusted efficiency measures by comparing a team's performance with the average performance of the opponents it faced. These are not intended to be a full rating system, but they provide a basic adjustment for differences in schedule difficulty.

In [415]:
team_season = (
    team_season
    .with_columns([
        (
            pl.col("off_epa_per_play") -
            pl.col("avg_opponent_def_epa_allowed")
        ).alias("adjusted_off_epa"),

        (
            pl.col("def_epa_per_play_allowed") -
            pl.col("avg_opponent_off_epa")
        ).alias("adjusted_def_epa_allowed")
    ])
)

In [416]:
team_season = (
    team_season
    .with_columns([
        (
            pl.col("off_epa_per_play") -
            pl.col("def_epa_per_play_allowed")
        ).alias("net_epa_per_play"),

        (
            pl.col("adjusted_off_epa") -
            pl.col("adjusted_def_epa_allowed")
        ).alias("adjusted_net_epa")
    ])
)

## Performance Consistency

Season averages can hide how much a team's performance changed from game to game.

I calculate game level offensive and defensive EPA and use their season long variation as a measure of consistency. This gives the model additional context about whether a team's overall performance was steady or driven by larger swings throughout the season.

In [417]:
game_team_epa = (
    pbp
    .filter(
        pl.col("posteam").is_not_null() &
        pl.col("defteam").is_not_null() &
        pl.col("epa").is_not_null()
    )
    .group_by([
        "season",
        "game_id",
        "posteam"
    ])
    .agg(
        pl.col("epa")
        .mean()
        .alias("game_off_epa")
    )
    .rename({
        "posteam": "team"
    })
)

game_team_def_epa = (
    pbp
    .filter(
        pl.col("posteam").is_not_null() &
        pl.col("defteam").is_not_null() &
        pl.col("epa").is_not_null()
    )
    .group_by([
        "season",
        "game_id",
        "defteam"
    ])
    .agg(
        pl.col("epa")
        .mean()
        .alias("game_def_epa_allowed")
    )
    .rename({
        "defteam": "team"
    })
)

In [418]:
team_consistency = (
    game_team_epa
    .join(
        game_team_def_epa,
        on=[
            "season",
            "game_id",
            "team"
        ],
        how="inner"
    )
    .group_by([
        "season",
        "team"
    ])
    .agg([
        pl.col("game_off_epa")
        .std()
        .alias("off_epa_game_sd"),

        pl.col("game_def_epa_allowed")
        .std()
        .alias("def_epa_game_sd")
    ])
)

In [419]:
team_season = (
    team_season
    .join(
        team_consistency,
        on=["season", "team"],
        how="left"
    )
)

## Expected Wins and Regression Indicators

A team's record does not always reflect how well it actually played.

Expected wins use points scored and points allowed to estimate the record a team's underlying performance would normally produce. I also compare expected and actual wins to identify teams that may have overperformed or underperformed their scoring profile.

In [420]:
PYTHAGOREAN_EXPONENT = 2.37

team_season = (
    team_season
    .with_columns([
        (
            (
                pl.col("points_for").cast(pl.Float64) ** PYTHAGOREAN_EXPONENT
            )
            /
            (
                (pl.col("points_for").cast(pl.Float64) ** PYTHAGOREAN_EXPONENT)
                +
                (pl.col("points_against").cast(pl.Float64) ** PYTHAGOREAN_EXPONENT)
            )
        ).alias("expected_win_pct")
    ])
    .with_columns([
        (
            pl.col("expected_win_pct") *
            pl.col("games")
        ).alias("expected_wins")
    ])
    .with_columns([
        (
            pl.col("wins") +
            (0.5 * pl.col("ties")) -
            pl.col("expected_wins")
        ).alias("wins_above_expected")
    ])
)

### One Score and Turnover Indicators

Close game results and turnovers can have a large impact on a team's record but are often less stable from year to year than underlying efficiency.

I keep these effects separate so the model can learn how much information they carry into the following season rather than assuming that all wins and losses are equally predictive.

In [421]:
team_season = (
    team_season
    .with_columns([
        (
            pl.col("one_score_wins") -
            (0.5 * pl.col("one_score_games"))
        ).alias("one_score_wins_above_500"),

        (
            pl.col("turnover_margin") /
            pl.col("games")
        ).alias("turnover_margin_per_game"),

        (
            pl.col("turnovers") /
            pl.col("games")
        ).alias("turnovers_per_game"),

        (
            pl.col("takeaways") /
            pl.col("games")
        ).alias("takeaways_per_game")
    ])
)

In [422]:
team_season = (
    team_season
    .sort([
        "team",
        "season"
    ])
    .with_columns([
        pl.col("wins_above_expected")
        .shift(1)
        .over("team")
        .alias("prev_wins_above_expected"),

        pl.col("one_score_win_pct")
        .shift(1)
        .over("team")
        .alias("prev_one_score_win_pct"),

        pl.col("one_score_wins_above_500")
        .shift(1)
        .over("team")
        .alias("prev_one_score_wins_above_500"),

        pl.col("turnover_margin_per_game")
        .shift(1)
        .over("team")
        .alias("prev_turnover_margin_per_game")
    ])
)

## Pass Protection

Pressure affects both sides of the ball.

In addition to measuring how often a defense creates sacks and quarterback hits, I measure how often each offense allows them. This adds context about pass protection and the ability of an offense to keep its quarterback clean.

In [423]:
offensive_pressure = (
    pbp
    .filter(
        pl.col("posteam").is_not_null() &
        pl.col("qb_dropback").is_not_null()
    )
    .group_by([
        "season",
        "posteam"
    ])
    .agg([
        pl.col("qb_dropback")
        .sum()
        .alias("off_dropbacks"),

        pl.col("sack")
        .sum()
        .alias("sacks_allowed"),

        pl.col("qb_hit")
        .sum()
        .alias("qb_hits_allowed")
    ])
    .with_columns([
        (
            pl.col("sacks_allowed") /
            pl.col("off_dropbacks")
        ).alias("sack_rate_allowed"),

        (
            pl.col("qb_hits_allowed") /
            pl.col("off_dropbacks")
        ).alias("qb_hit_rate_allowed")
    ])
    .rename({
        "posteam": "team"
    })
)

In [424]:
team_season = (
    team_season
    .join(
        offensive_pressure,
        on=["season", "team"],
        how="left"
    )
)

## Drive-Level Efficiency

Play level efficiency does not always translate directly into points.

I also measure how often teams turn possessions into scoring and how many points they produce or allow per drive. These features provide a possession level view of offensive and defensive performance alongside EPA.

In [425]:
drives = (
    pbp
    .filter(
        pl.col("posteam").is_not_null() &
        pl.col("defteam").is_not_null() &
        pl.col("drive").is_not_null()
    )
    .group_by([
        "season",
        "game_id",
        "posteam",
        "defteam",
        "drive"
    ])
    .agg([
        pl.col("posteam_score")
        .drop_nulls()
        .first()
        .alias("start_score"),

        pl.col("posteam_score_post")
        .drop_nulls()
        .last()
        .alias("end_score")
    ])
    .with_columns(
        (
            pl.col("end_score") -
            pl.col("start_score")
        ).alias("drive_points")
    )
    .filter(
        pl.col("drive_points").is_between(0, 8)
    )
)

In [426]:
off_drive_features = (
    drives
    .group_by([
        "season",
        "posteam"
    ])
    .agg([
        pl.len().alias("off_drives"),

        pl.col("drive_points")
        .sum()
        .alias("off_drive_points"),

        pl.col("drive_points")
        .mean()
        .alias("points_per_drive"),

        (pl.col("drive_points") > 0)
        .mean()
        .alias("scoring_drive_rate")
    ])
    .rename({
        "posteam": "team"
    })
)

def_drive_features = (
    drives
    .group_by([
        "season",
        "defteam"
    ])
    .agg([
        pl.len().alias("def_drives"),

        pl.col("drive_points")
        .sum()
        .alias("def_drive_points_allowed"),

        pl.col("drive_points")
        .mean()
        .alias("points_per_drive_allowed"),

        (pl.col("drive_points") > 0)
        .mean()
        .alias("scoring_drive_rate_allowed")
    ])
    .rename({
        "defteam": "team"
    })
)

In [427]:
team_season = (
    team_season
    .join(
        off_drive_features,
        on=["season", "team"],
        how="left"
    )
    .join(
        def_drive_features,
        on=["season", "team"],
        how="left"
    )
)

In [428]:
team_season.filter(
    pl.col("season") == 2025
).select([
    "team",
    "off_drives",
    "points_per_drive",
    "scoring_drive_rate",
    "def_drives",
    "points_per_drive_allowed",
    "scoring_drive_rate_allowed"
]).sort(
    "points_per_drive",
    descending=True
)

team,off_drives,points_per_drive,scoring_drive_rate,def_drives,points_per_drive_allowed,scoring_drive_rate_allowed
str,u32,f64,f64,u32,f64,f64
"""LA""",181,2.78453,0.469613,177,1.79661,0.344633
"""BUF""",175,2.668571,0.451429,177,2.062147,0.372881
"""IND""",173,2.653179,0.49711,176,2.255682,0.443182
"""NE""",176,2.613636,0.460227,173,1.768786,0.317919
"""DAL""",177,2.559322,0.480226,178,2.859551,0.511236
…,…,…,…,…,…,…
"""NO""",176,1.619318,0.329545,183,2.005464,0.398907
"""NYJ""",184,1.505435,0.298913,186,2.666667,0.489247
"""TEN""",183,1.42623,0.295082,184,2.445652,0.451087


### Red-Zone Touchdown Rate

Red-zone EPA measures efficiency near the goal line, but I also want to track how often red-zone possessions actually end in touchdowns.

I identify drives that reached the opponent's 20-yard line and measure the share of those drives that finished with a touchdown.

In [429]:
red_zone_drives = (
    pbp
    .filter(
        pl.col("posteam").is_not_null() &
        pl.col("defteam").is_not_null() &
        pl.col("fixed_drive").is_not_null()
    )
    .group_by([
        "season",
        "game_id",
        "posteam",
        "defteam",
        "fixed_drive"
    ])
    .agg([
        (
            pl.col("yardline_100")
            .filter(pl.col("yardline_100").is_not_null())
            .min()
        ).alias("closest_yardline"),

        pl.col("fixed_drive_result")
        .drop_nulls()
        .last()
        .alias("drive_result")
    ])
    .filter(
        pl.col("closest_yardline") <= 20
    )
    .with_columns(
        (pl.col("drive_result") == "Touchdown")
        .cast(pl.Int8)
        .alias("red_zone_td")
    )
)

In [430]:
off_red_zone_td = (
    red_zone_drives
    .group_by([
        "season",
        "posteam"
    ])
    .agg([
        pl.len().alias("red_zone_drives"),
        pl.col("red_zone_td").sum().alias("red_zone_tds"),
        pl.col("red_zone_td").mean().alias("red_zone_td_rate")
    ])
    .rename({
        "posteam": "team"
    })
)

def_red_zone_td = (
    red_zone_drives
    .group_by([
        "season",
        "defteam"
    ])
    .agg([
        pl.len().alias("red_zone_drives_allowed"),
        pl.col("red_zone_td").sum().alias("red_zone_tds_allowed"),
        pl.col("red_zone_td").mean().alias("red_zone_td_rate_allowed")
    ])
    .rename({
        "defteam": "team"
    })
)

In [431]:
team_season = (
    team_season
    .join(
        off_red_zone_td,
        on=["season", "team"],
        how="left"
    )
    .join(
        def_red_zone_td,
        on=["season", "team"],
        how="left"
    )
)

In [432]:
team_season.filter(
    pl.col("season") == 2025
).select([
    "team",
    "red_zone_drives",
    "red_zone_tds",
    "red_zone_td_rate",
    "red_zone_drives_allowed",
    "red_zone_tds_allowed",
    "red_zone_td_rate_allowed"
]).sort(
    "red_zone_td_rate",
    descending=True
)

team,red_zone_drives,red_zone_tds,red_zone_td_rate,red_zone_drives_allowed,red_zone_tds_allowed,red_zone_td_rate_allowed
str,u32,i64,f64,u32,i64,f64
"""PHI""",60,43,0.716667,56,33,0.589286
"""CIN""",67,48,0.716418,81,51,0.62963
"""BUF""",85,60,0.705882,63,43,0.68254
"""DET""",80,55,0.6875,69,47,0.681159
"""IND""",77,52,0.675325,74,40,0.540541
…,…,…,…,…,…,…
"""CLE""",49,26,0.530612,65,40,0.615385
"""LAC""",69,36,0.521739,64,32,0.5
"""NYJ""",54,27,0.5,82,56,0.682927


## Special Teams

Special teams can change field position and scoring even though it makes up a smaller share of total plays.

I keep this section simple and focus on field goal performance and return production so special teams has some representation in the historical team feature set without building a separate rating model.

In [433]:
special_teams = (
    player_stats
    .group_by([
        "season",
        pl.col("recent_team").alias("team")
    ])
    .agg([
        pl.col("fg_made").sum().alias("fg_made"),
        pl.col("fg_att").sum().alias("fg_attempts"),

        pl.col("punt_returns").sum().alias("punt_returns"),
        pl.col("punt_return_yards").sum().alias("punt_return_yards"),

        pl.col("kickoff_returns").sum().alias("kickoff_returns"),
        pl.col("kickoff_return_yards").sum().alias("kickoff_return_yards")
    ])
    .with_columns([
        pl.when(pl.col("fg_attempts") > 0)
        .then(pl.col("fg_made") / pl.col("fg_attempts"))
        .otherwise(None)
        .alias("fg_pct"),

        pl.when(pl.col("punt_returns") > 0)
        .then(pl.col("punt_return_yards") / pl.col("punt_returns"))
        .otherwise(None)
        .alias("punt_return_avg"),

        pl.when(pl.col("kickoff_returns") > 0)
        .then(pl.col("kickoff_return_yards") / pl.col("kickoff_returns"))
        .otherwise(None)
        .alias("kickoff_return_avg")
    ])
)

In [434]:
team_season = (
    team_season
    .join(
        special_teams,
        on=["season", "team"],
        how="left"
    )
)

## Schedule and Game Context

Team performance can vary depending on where games are played and how much recovery time a team has between games.

I summarize home and road performance along with season level rest patterns. These features provide additional context around a team's historical results without attempting to measure future schedule strength here.

In [435]:
home_road_features = (
    team_games
    .group_by([
        "season",
        "team"
    ])
    .agg([
        (pl.col("home_game") == 1)
        .sum()
        .alias("home_games"),

        (pl.col("home_game") == 0)
        .sum()
        .alias("road_games"),

        pl.col("win")
        .filter(pl.col("home_game") == 1)
        .sum()
        .alias("home_wins"),

        pl.col("win")
        .filter(pl.col("home_game") == 0)
        .sum()
        .alias("road_wins"),

        pl.col("point_diff")
        .filter(pl.col("home_game") == 1)
        .mean()
        .alias("home_avg_point_diff"),

        pl.col("point_diff")
        .filter(pl.col("home_game") == 0)
        .mean()
        .alias("road_avg_point_diff")
    ])
    .with_columns([
        (
            pl.col("home_wins") /
            pl.col("home_games")
        ).alias("home_win_pct"),

        (
            pl.col("road_wins") /
            pl.col("road_games")
        ).alias("road_win_pct")
    ])
)

In [436]:
home_rest = (
    schedule
    .select([
        "game_id",
        "season",
        "home_team",
        "home_rest"
    ])
    .rename({
        "home_team": "team",
        "home_rest": "rest_days"
    })
)

away_rest = (
    schedule
    .select([
        "game_id",
        "season",
        "away_team",
        "away_rest"
    ])
    .rename({
        "away_team": "team",
        "away_rest": "rest_days"
    })
)

team_rest = pl.concat([
    home_rest,
    away_rest
])

In [437]:
rest_features = (
    team_rest
    .group_by([
        "season",
        "team"
    ])
    .agg([
        pl.col("rest_days")
        .mean()
        .alias("avg_rest_days"),

        (pl.col("rest_days") < 7)
        .sum()
        .alias("short_rest_games"),

        (pl.col("rest_days") > 7)
        .sum()
        .alias("extended_rest_games")
    ])
)

In [438]:
team_season = (
    team_season
    .join(
        home_road_features,
        on=["season", "team"],
        how="left"
    )
    .join(
        rest_features,
        on=["season", "team"],
        how="left"
    )
)

## Final Feature Validation

Before saving the historical team feature set, I run a final set of checks for row structure, duplicate team seasons, missing values, and feature coverage.

The goal is to make sure the output contains one reliable row per team season before it is used by the projection and modeling notebooks.

In [441]:
print("Final shape:", team_season.shape)

print(
    "Seasons:",
    team_season["season"].min(),
    "to",
    team_season["season"].max()
)

print(
    "Duplicate team-seasons:",
    team_season.select([
        "season",
        "team"
    ]).is_duplicated().sum()
)

print(
    team_season
    .group_by("season")
    .agg(
        pl.len().alias("team_rows")
    )
    .sort("season")
)

Final shape: (352, 156)
Seasons: 2015 to 2025
Duplicate team-seasons: 0
shape: (11, 2)
┌────────┬───────────┐
│ season ┆ team_rows │
│ ---    ┆ ---       │
│ i32    ┆ u32       │
╞════════╪═══════════╡
│ 2015   ┆ 32        │
│ 2016   ┆ 32        │
│ 2017   ┆ 32        │
│ 2018   ┆ 32        │
│ 2019   ┆ 32        │
│ …      ┆ …         │
│ 2021   ┆ 32        │
│ 2022   ┆ 32        │
│ 2023   ┆ 32        │
│ 2024   ┆ 32        │
│ 2025   ┆ 32        │
└────────┴───────────┘


In [442]:
null_summary = (
    pl.DataFrame({
        "feature": team_season.columns,
        "nulls": [
            team_season[col].null_count()
            for col in team_season.columns
        ]
    })
    .filter(
        pl.col("nulls") > 0
    )
    .sort(
        "nulls",
        descending=True
    )
)

print(null_summary)

shape: (25, 2)
┌───────────────────────────────┬───────┐
│ feature                       ┆ nulls │
│ ---                           ┆ ---   │
│ str                           ┆ i64   │
╞═══════════════════════════════╪═══════╡
│ off_epa_3yr_avg               ┆ 64    │
│ def_epa_allowed_3yr_avg       ┆ 64    │
│ offense_continuity            ┆ 32    │
│ defense_continuity            ┆ 32    │
│ overall_continuity            ┆ 32    │
│ …                             ┆ …     │
│ def_epa_allowed_2yr_avg       ┆ 32    │
│ prev_wins_above_expected      ┆ 32    │
│ prev_one_score_win_pct        ┆ 32    │
│ prev_one_score_wins_above_500 ┆ 32    │
│ prev_turnover_margin_per_game ┆ 32    │
└───────────────────────────────┴───────┘


In [443]:
print(
    null_summary
    .sort(["nulls", "feature"], descending=[True, False])
    .to_dicts()
)

[{'feature': 'def_epa_allowed_3yr_avg', 'nulls': 64}, {'feature': 'off_epa_3yr_avg', 'nulls': 64}, {'feature': 'def_epa_allowed_2yr_avg', 'nulls': 32}, {'feature': 'def_epa_allowed_change', 'nulls': 32}, {'feature': 'defense_continuity', 'nulls': 32}, {'feature': 'front_seven_continuity', 'nulls': 32}, {'feature': 'off_epa_2yr_avg', 'nulls': 32}, {'feature': 'off_epa_change', 'nulls': 32}, {'feature': 'offense_continuity', 'nulls': 32}, {'feature': 'ol_continuity', 'nulls': 32}, {'feature': 'overall_continuity', 'nulls': 32}, {'feature': 'passing_yards_continuity', 'nulls': 32}, {'feature': 'prev_def_epa_per_play_allowed', 'nulls': 32}, {'feature': 'prev_off_epa_per_play', 'nulls': 32}, {'feature': 'prev_one_score_win_pct', 'nulls': 32}, {'feature': 'prev_one_score_wins_above_500', 'nulls': 32}, {'feature': 'prev_turnover_margin_per_game', 'nulls': 32}, {'feature': 'prev_wins', 'nulls': 32}, {'feature': 'prev_wins_above_expected', 'nulls': 32}, {'feature': 'qb_continuity', 'nulls': 32}

In [444]:
for i, col in enumerate(team_season.columns, start=1):
    print(f"{i:>3}. {col}")

  1. season
  2. team
  3. games
  4. wins
  5. losses
  6. ties
  7. points_for
  8. points_against
  9. point_diff
 10. win_pct
 11. points_for_per_game
 12. points_against_per_game
 13. point_diff_per_game
 14. win_loss_diff
 15. positive_point_diff
 16. one_score_games
 17. one_score_wins
 18. one_score_losses
 19. one_score_ties
 20. one_score_win_pct
 21. off_plays
 22. off_epa_per_play
 23. off_success_rate
 24. off_pass_epa_per_play
 25. off_rush_epa_per_play
 26. def_plays
 27. def_epa_per_play_allowed
 28. def_success_rate_allowed
 29. def_pass_epa_per_play_allowed
 30. def_rush_epa_per_play_allowed
 31. off_explosive_rate
 32. off_explosive_pass_rate
 33. off_explosive_rush_rate
 34. def_explosive_rate_allowed
 35. def_explosive_pass_rate_allowed
 36. def_explosive_rush_rate_allowed
 37. interceptions_thrown
 38. fumbles_lost
 39. turnovers
 40. def_interceptions
 41. def_fumble_recoveries
 42. takeaways
 43. def_pass_plays
 44. sacks
 45. qb_hits
 46. sack_rate
 47. qb_hit_

In [445]:
# Final validation of historical team feature dataset

expected_seasons = list(range(2015, 2026))

assert team_season.height == 352, \
    f"Expected 352 team-seasons, found {team_season.height}"

assert team_season.select(
    ["season", "team"]
).is_duplicated().sum() == 0, \
    "Duplicate team-season rows found"

season_counts = (
    team_season
    .group_by("season")
    .agg(
        pl.len().alias("teams")
    )
    .sort("season")
)

assert season_counts["season"].to_list() == expected_seasons, \
    "Unexpected seasons in dataset"

assert season_counts["teams"].to_list() == [32] * 11, \
    "A season does not contain exactly 32 teams"

print("Historical team feature validation passed.")
print(f"Rows: {team_season.height}")
print(f"Features: {team_season.width}")
print(
    f"Seasons: {team_season['season'].min()}–"
    f"{team_season['season'].max()}"
)

Historical team feature validation passed.
Rows: 352
Features: 156
Seasons: 2015–2025


## Save Historical Team Features

The completed team season feature table is saved as the primary historical modeling dataset.

Each row represents one NFL team in one season and combines team performance, efficiency, roster continuity, injuries, regression indicators, opponent adjusted performance, situational football metrics, and schedule context.

In [446]:
team_season = (
    team_season
    .sort([
        "season",
        "team"
    ])
)

output_file = (
    PROCESSED_DIR /
    "historical_team_features.parquet"
)

team_season.write_parquet(
    output_file
)

print(f"Saved: {output_file}")
print(f"Shape: {team_season.shape}")

Saved: c:\Users\efriedman\Desktop\NFL-Season-Projections\data\processed\historical_team_features.parquet
Shape: (352, 156)


In [447]:
# Confirm saved dataset can be loaded successfully

historical_team_features = pl.read_parquet(
    PROCESSED_DIR /
    "historical_team_features.parquet"
)

print(
    "Loaded historical features:",
    historical_team_features.shape
)

historical_team_features.tail()

Loaded historical features: (352, 156)


season,team,games,wins,losses,ties,points_for,points_against,point_diff,win_pct,points_for_per_game,points_against_per_game,point_diff_per_game,win_loss_diff,positive_point_diff,one_score_games,one_score_wins,one_score_losses,one_score_ties,one_score_win_pct,off_plays,off_epa_per_play,off_success_rate,off_pass_epa_per_play,off_rush_epa_per_play,def_plays,def_epa_per_play_allowed,def_success_rate_allowed,def_pass_epa_per_play_allowed,def_rush_epa_per_play_allowed,off_explosive_rate,off_explosive_pass_rate,off_explosive_rush_rate,def_explosive_rate_allowed,def_explosive_pass_rate_allowed,def_explosive_rush_rate_allowed,interceptions_thrown,…,qb_hits_allowed,sack_rate_allowed,qb_hit_rate_allowed,off_drives,off_drive_points,points_per_drive,scoring_drive_rate,def_drives,def_drive_points_allowed,points_per_drive_allowed,scoring_drive_rate_allowed,red_zone_drives,red_zone_tds,red_zone_td_rate,red_zone_drives_allowed,red_zone_tds_allowed,red_zone_td_rate_allowed,fg_made,fg_attempts,punt_returns,punt_return_yards,kickoff_returns,kickoff_return_yards,fg_pct,punt_return_avg,kickoff_return_avg,home_games,road_games,home_wins,road_wins,home_avg_point_diff,road_avg_point_diff,home_win_pct,road_win_pct,avg_rest_days,short_rest_games,extended_rest_games
i32,str,u32,i64,i64,i64,i32,i32,i32,f64,f64,f64,f64,i64,i8,u32,i64,i64,i64,f64,u32,f64,f64,f64,f64,u32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,u32,f64,f64,f64,u32,f64,f64,f64,u32,i64,f64,u32,i64,f64,i32,i32,i32,i32,i32,i32,f64,f64,f64,u32,u32,i64,i64,f64,f64,f64,f64,f64,u32,u32
2025,"""SEA""",17,14,3,0,483,292,191,0.823529,28.411765,17.176471,11.235294,11,1,9,6,3,0,0.666667,997,0.032795,0.46339,0.112228,-0.049065,1061,-0.115932,0.396795,-0.097349,-0.145322,0.112337,0.112648,0.112016,0.067861,0.067692,0.068127,15.0,…,60.0,0.051923,0.115385,187,441.0,2.358289,0.465241,191,279.0,1.460733,0.282723,78,45,0.576923,58,29,0.5,41,48,39,577,55,1499,0.854167,14.794872,27.254545,8,9,6,8,10.375,12.0,0.75,0.888889,7.352941,3,4
2025,"""SF""",17,12,5,0,437,371,66,0.705882,25.705882,21.823529,3.882353,7,1,6,5,1,0,0.833333,1065,0.087955,0.482629,0.158693,-0.004373,1019,0.070436,0.484789,0.112152,0.009968,0.09108,0.084577,0.099567,0.080471,0.074627,0.088942,16.0,…,82.0,0.042925,0.130366,171,430.0,2.51462,0.473684,168,356.0,2.119048,0.380952,72,48,0.666667,70,42,0.6,45,52,25,291,51,1385,0.865385,11.64,27.156863,8,9,5,7,1.0,6.444444,0.625,0.777778,7.352941,4,4
2025,"""TB""",17,8,9,0,380,411,-31,0.470588,22.352941,24.176471,-1.823529,-1,0,12,6,6,0,0.5,1065,-0.008881,0.423474,-0.026404,0.013815,990,0.013279,0.448485,0.06803,-0.072395,0.096714,0.091514,0.103448,0.094949,0.09106,0.101036,11.0,…,56.0,0.059006,0.086957,183,364.0,1.989071,0.387978,180,390.0,2.166667,0.388889,67,39,0.58209,65,46,0.707692,32,38,26,291,61,1484,0.842105,11.192308,24.327869,8,9,4,4,0.25,-3.666667,0.5,0.444444,7.352941,4,4
2025,"""TEN""",17,3,14,0,284,478,-194,0.176471,16.705882,28.117647,-11.411765,-11,0,7,2,5,0,0.285714,995,-0.159669,0.373869,-0.220785,-0.057316,992,0.101346,0.451613,0.167288,0.016563,0.081407,0.065811,0.107527,0.107863,0.103943,0.112903,8.0,…,110.0,0.086687,0.170279,183,261.0,1.42623,0.295082,184,450.0,2.445652,0.451087,44,25,0.568182,76,51,0.671053,28,35,24,399,65,1669,0.8,16.625,25.676923,9,8,1,2,-9.111111,-14.0,0.111111,0.25,7.411765,0,1
2025,"""WAS""",17,5,12,0,356,451,-95,0.294118,20.941176,26.529412,-5.588235,-7,0,7,2,5,0,0.285714,979,0.006479,0.456588,-0.005508,0.019198,1073,0.143335,0.476235,0.200874,0.074876,0.102145,0.079365,0.126316,0.109972,0.090909,0.132653,12.0,…,77.0,0.065141,0.135563,167,341.0,2.041916,0.371257,170,444.0,2.611765,0.470588,58,39,0.672414,74,51,0.689189,19,23,26,315,71,1918,0.826087,12.115385,27.014085,8,9,2,3,-4.25,-6.777778,0.25,0.333333,7.411765,5,5
